# Woche 08: Indexierung mit k-mer-Index und Hashtabelle

**Block B · Woche 08 · Selbststudium · Starter (Skeleton)**

Dieses Notebook setzt die Theorie aus dem Skriptum *„Indexierung: k-mer-Index, Binärsuche und Hashtabelle“*
(Woche 08) in Python um. Es ergänzt die Pflichtübung Ü2 (FASTA parsen) und ist in etwa 100 Minuten lösbar. Alle
Aufgaben lassen sich mit dem Skriptum und den Folien der Woche lösen; Abschnittsnummern verweisen auf das Skriptum.

## Lernziele
- Einen **k-mer-Index** für einen Text T als Dictionary aufbauen (Skriptum 3)
- Den Index mit einem Muster P **abfragen** und jeden Index-Treffer **verifizieren** (Skriptum 3)
- Denselben Index als **sortierte Liste** mit **Binärsuche** (`bisect`) umsetzen (Skriptum 4.1)
- Verstehen, wie eine **Hashtabelle** k-mers auf Buckets verteilt und was eine **Kollision** ist (Skriptum 4.2)
- Eine **Index-Variante** programmieren, die nur jedes n-te k-mer speichert (Skriptum 5.1)
- Am phiX-Genom messen, wie k und die Variante die Zahl der Index-Treffer verändern (Skriptum 6)

## Zu implementierende Funktionen
| Teil | Funktion(en) | Skriptum | Zeit |
|---|---|---|---|
| 1 | `build_kmer_index` | 3.1 | ≈ 15 min |
| 2 | `query_kmer_index` | 3.2, 3.3 | ≈ 20 min |
| 3 | `build_sorted_index`, `lookup_sorted_index` | 4.1 | ≈ 20 min |
| 4 | `kmer_to_number`, `build_hash_table`, `lookup_hash_table` | 4.2 | ≈ 15 min |
| 5 | `build_sparse_index`, `query_sparse_index` | 5.1 | ≈ 20 min |
| 6 | Messungen ausführen, Fragen beantworten | 6 | ≈ 10 min |
| Bonus | `subsequence`, `build_subseq_index`, `query_subseq_index` | 5.2 | optional |

## Begriffe im Code
- **k-mer**: Teilstring der Länge k. Ein Text der Länge n hat n − k + 1 k-mers (mit Wiederholungen).
- **Index-Treffer (index hit)**: Ein Offset, den der Index für ein k-mer von P liefert. Das ist nur ein **Kandidat**.
- **Verifikation**: Prüfen, ob an einem Kandidaten wirklich ganz P im Text steht. Erst dann ist es ein Vorkommen
  (occurrence).

**So arbeitest du:** Setup-Zelle ausführen, dann Teil für Teil die Funktionsrümpfe mit `NotImplementedError` durch
deinen Code ersetzen und die Test-Zelle (`%%ipytest`) ausführen, bis alle Tests grün sind. Der Code bleibt
englisch (Bezeichner, Ausgaben); Kommentare darfst du deutsch schreiben.

## 0. Setup
Führe diese Zelle zuerst aus. Fehlende Pakete werden einmalig installiert. Die Datendateien `phix.fa`,
`ERR266411_1.first1000.fastq` und `lambda_virus.fa` werden im aktuellen Ordner, in `daten`, in `../daten` und in `..`
gesucht.

In [ ]:
# Benötigte Pakete (werden bei Bedarf einmalig installiert)
import importlib, subprocess, sys
for package in ["ipytest", "pytest"]:
    try:
        importlib.import_module(package)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package])

import bisect
import os
import random
import time

import pytest
import ipytest
ipytest.autoconfig()

DATA_DIRS = [".", "daten", os.path.join("..", "daten"), ".."]


def data_path(filename: str) -> str:
    """Return the path of a data file, searching the folders in DATA_DIRS."""
    for folder in DATA_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{filename} not found in {DATA_DIRS}. Copy it next to this notebook.")


for f in ["phix.fa", "ERR266411_1.first1000.fastq", "lambda_virus.fa"]:
    print(f"{f}: {data_path(f)}")

### Werkzeuge aus Woche 6 (fertig, nur ausführen)
Diese Funktionen kennst du aus Woche 6 und 7. `naive_with_counts` ist die Messlatte: Jede Index-Suche muss genau
dieselben Offsets liefern wie der naive Algorithmus.

In [ ]:
def read_genome(filename: str) -> str:
    """Read a FASTA file and return the sequence without header lines."""
    genome = ""
    with open(filename) as f:
        for line in f:
            if not line.startswith(">"):
                genome += line.rstrip()
    return genome


def read_fastq(filename: str) -> tuple[list[str], list[str]]:
    """Read a FASTQ file and return (sequences, qualities)."""
    sequences, qualities = [], []
    with open(filename) as fh:
        while True:
            fh.readline()                   # name line (@...)
            seq = fh.readline().rstrip()
            fh.readline()                   # plus line
            qual = fh.readline().rstrip()
            if len(seq) == 0:
                break
            sequences.append(seq)
            qualities.append(qual)
    return sequences, qualities


def naive_with_counts(p: str, t: str) -> tuple[list[int], int, int]:
    """Naive exact matching. Returns (occurrences, n_alignments, n_comparisons)."""
    occurrences, n_alignments, n_comparisons = [], 0, 0
    for i in range(len(t) - len(p) + 1):
        n_alignments += 1
        match = True
        for j in range(len(p)):
            n_comparisons += 1
            if t[i + j] != p[j]:
                match = False
                break
        if match:
            occurrences.append(i)
    return occurrences, n_alignments, n_comparisons


def naive(p: str, t: str) -> list[int]:
    """All offsets where p occurs in t (naive algorithm)."""
    return naive_with_counts(p, t)[0]


genome = read_genome(data_path("phix.fa"))
reads, _ = read_fastq(data_path("ERR266411_1.first1000.fastq"))
lambda_genome = read_genome(data_path("lambda_virus.fa"))
print(f"phiX: {len(genome)} nt | reads: {len(reads)} | lambda: {len(lambda_genome)} nt")

## 1. Einen k-mer-Index bauen (Skriptum 3.1)

Ein Index ordnet jedem k-mer des Textes T die Offsets zu, an denen es in T beginnt. Langmeads Beispiel mit
T = CGTGCGTGCTT und k = 5 (Skriptum Tab. 1):

| k-mer | Offsets |
|---|---|
| CGTGC | 0, 4 |
| GCGTG | 3 |
| GTGCG | 1 |
| GTGCT | 5 |
| TGCGT | 2 |
| TGCTT | 6 |

In Python passt dafür ein **Dictionary**: Schlüssel ist das k-mer, Wert die Liste der Offsets. Ein Python-Dictionary
ist intern eine Hashtabelle (Teil 4).

**Aufgabe 1.1:** Schreibe `build_kmer_index(t, k)`. Gehe alle Offsets `i` von 0 bis `len(t) - k` durch, schneide das
k-mer `t[i:i + k]` aus und hänge `i` an die Liste dieses k-mers an. Die Offsets in jeder Liste sind dadurch
automatisch aufsteigend sortiert. Ist T kürzer als k, ist das Ergebnis ein leeres Dictionary.

*Tipps:* `if kmer not in index: index[kmer] = []` legt eine neue Liste an. Kürzer geht es mit
`index.setdefault(kmer, []).append(i)`.

In [ ]:
def build_kmer_index(t: str, k: int) -> dict[str, list[int]]:
    """Map every k-mer of t to the sorted list of offsets where it starts."""
    # Tipp: for i in range(len(t) - k + 1): k-mer ausschneiden, Offset an seine Liste anhängen
    raise NotImplementedError("TODO: implement build_kmer_index")

In [ ]:
# Langmeads Beispiel (Skriptum 3.1, Tab. 1): alphabetisch ausgeben
example_index = build_kmer_index("CGTGCGTGCTT", 5)
for kmer in sorted(example_index):
    print(kmer, example_index[kmer])

# Wie viele verschiedene k-mers hat das phiX-Genom?
for k in [4, 8, 12]:
    phix_index = build_kmer_index(genome, k)
    n_entries = sum(len(offsets) for offsets in phix_index.values())
    print(f"k={k:2d}: {len(phix_index):5d} distinct k-mers, {n_entries} entries, 4^k = {4 ** k}")

In [ ]:
%%ipytest

def test_build_kmer_index_slide_example():
    assert build_kmer_index("CGTGCGTGCTT", 5) == {
        "CGTGC": [0, 4], "GTGCG": [1], "TGCGT": [2],
        "GCGTG": [3], "GTGCT": [5], "TGCTT": [6],
    }


def test_build_kmer_index_python_dictionary_slide():
    # Dictionary von Langmeads Folie „Python dictionary“ (Skriptum 4.2)
    assert build_kmer_index("GTGCGTGTGGGGG", 3) == {
        "GTG": [0, 4, 6], "TGC": [1], "GCG": [2], "CGT": [3],
        "TGT": [5], "TGG": [7], "GGG": [8, 9, 10],
    }


def test_build_kmer_index_edge_cases():
    assert build_kmer_index("ACG", 5) == {}
    assert build_kmer_index("ACGT", 4) == {"ACGT": [0]}
    assert build_kmer_index("AAAA", 2) == {"AA": [0, 1, 2]}


def test_build_kmer_index_phix():
    phix_index = build_kmer_index(read_genome(data_path("phix.fa")), 8)
    assert len(phix_index) == 4921                                    # verschiedene 8-mers
    assert sum(len(v) for v in phix_index.values()) == 5386 - 8 + 1   # n - k + 1 Einträge

## 2. Den Index abfragen und verifizieren (Skriptum 3.2 und 3.3)

Um P in T zu finden, schlägt man das **erste k-mer von P** im Index nach. Jeder gelieferte Offset `i` ist ein
**Index-Treffer**: Dort beginnt in T dasselbe k-mer wie in P. Ob dort auch der Rest von P steht, weiß man noch nicht.
Deshalb wird jeder Index-Treffer **verifiziert**: Steht ab `i` wirklich ganz P im Text?

Beispiel aus Langmeads Folien mit dem Index aus Teil 1 (Skriptum 3.2):
- P = GCGTGC: erstes 5-mer GCGTG → Index-Treffer 3 → Verifikation erfolgreich → P kommt bei Offset 3 vor.
- P = GCGTGA: Index-Treffer 3 → Verifikation scheitert am letzten Zeichen → P kommt nicht vor.
- P = GCGTAC: GCGTA steht nicht im Index → keine Index-Treffer → P kommt nicht vor.

**Aufgabe 2.1:** Schreibe `query_kmer_index(p, t, index, k)`. Rückgabe ist ein Tupel
`(occurrences, n_hits)`: die Liste der verifizierten Offsets und die Zahl der Index-Treffer. Ist P kürzer als k,
kann man das erste k-mer nicht bilden: Löse dann `ValueError` aus.

*Tipps:* `index.get(kmer, [])` liefert eine leere Liste, wenn das k-mer fehlt. Für die Verifikation reicht
`t[i:i + len(p)] == p`. Überlege, warum ein Index-Treffer nahe am Textende kein Problem macht.

In [ ]:
def query_kmer_index(p: str, t: str, index: dict[str, list[int]], k: int) -> tuple[list[int], int]:
    """Find p in t with a k-mer index. Returns (verified occurrences, number of index hits)."""
    # Tipp: 1) len(p) < k -> ValueError  2) Treffer für p[:k] holen  3) jeden Treffer verifizieren
    raise NotImplementedError("TODO: implement query_kmer_index")

In [ ]:
t_example = "CGTGCGTGCTT"
example_index = build_kmer_index(t_example, 5)
for p in ["GCGTGC", "GCGTGA", "GCGTAC"]:
    print(p, query_kmer_index(p, t_example, example_index, 5))

# Beispiel aus dem Kursnotebook 2.02_SubstringIndex (k = 4)
t_course = "ACTTGGAGATCTTTGAGGCTAGGTATTCGGGATCGAAGCTCATTTCGGGGATCGATTACGATATGGTGGGTATTCGGGA"
print(query_kmer_index("GGTATTCGGGA", t_course, build_kmer_index(t_course, 4), 4))

In [ ]:
%%ipytest

T_COURSE = "ACTTGGAGATCTTTGAGGCTAGGTATTCGGGATCGAAGCTCATTTCGGGGATCGATTACGATATGGTGGGTATTCGGGA"


def test_query_slide_examples():
    t = "CGTGCGTGCTT"
    index = build_kmer_index(t, 5)
    assert query_kmer_index("GCGTGC", t, index, 5) == ([3], 1)
    assert query_kmer_index("GCGTGA", t, index, 5) == ([], 1)       # Treffer, aber Verifikation scheitert
    assert query_kmer_index("GCGTAC", t, index, 5) == ([], 0)       # k-mer nicht im Index


def test_query_course_notebook_202():
    assert query_kmer_index("GGTATTCGGGA", T_COURSE, build_kmer_index(T_COURSE, 4), 4)[0] == [21, 68]
    assert query_kmer_index("TTTT", T_COURSE, build_kmer_index(T_COURSE, 4), 4)[0] == []


def test_query_pattern_equal_to_k_and_overlaps():
    t = "AAAAAA"
    assert query_kmer_index("AAA", t, build_kmer_index(t, 3), 3) == ([0, 1, 2, 3], 4)
    assert query_kmer_index("AAAA", t, build_kmer_index(t, 2), 2) == ([0, 1, 2], 5)   # Treffer 3, 4 zu nah am Ende


def test_query_too_short_pattern():
    with pytest.raises(ValueError):
        query_kmer_index("AC", "ACGTACGT", build_kmer_index("ACGTACGT", 3), 3)


def test_query_equals_naive_on_random_dna():
    rng = random.Random(8)
    for _ in range(200):
        t = "".join(rng.choice("ACGT") for _ in range(300))
        k = rng.randint(2, 6)
        index = build_kmer_index(t, k)
        if rng.random() < 0.5:
            start = rng.randint(0, 280)
            p = t[start:start + rng.randint(k, 15)]
        else:
            p = "".join(rng.choice("AC") for _ in range(rng.randint(k, 10)))
        assert query_kmer_index(p, t, index, k)[0] == naive(p, t)

## 3. Index als sortierte Liste mit Binärsuche (Skriptum 4.1)

Statt eines Dictionaries kann man alle Paare `(k-mer, offset)` in einer **Liste** speichern und sie
**alphabetisch sortieren**. Gleiche k-mers liegen dann nebeneinander. Langmeads Beispiel T = GTGCGTGTGGGGG, k = 3:

`[('CGT', 3), ('GCG', 2), ('GGG', 8), ('GGG', 9), ('GGG', 10), ('GTG', 0), ('GTG', 4), ('GTG', 6), ('TGC', 1), ('TGG', 7), ('TGT', 5)]`

In einer sortierten Liste findet die **Binärsuche** einen Eintrag mit etwa log₂(n) Halbierungen. Python bringt sie im
Modul `bisect` mit: `bisect.bisect_left(a, x)` liefert die **linkeste** Position, an der man `x` einfügen könnte,
ohne die Sortierung zu zerstören. Beispiel von Langmeads Folie:

```python
a = [1, 3, 3, 6, 8, 8, 9, 10]
bisect.bisect_left(a, 2)   # 1
bisect.bisect_left(a, 4)   # 3
bisect.bisect_left(a, 8)   # 4  (erste 8)
```

**Aufgabe 3.1:** Schreibe `build_sorted_index(t, k)`: Liste aller Tupel `(t[i:i + k], i)`, sortiert.

**Aufgabe 3.2:** Schreibe `lookup_sorted_index(index, kmer)`: alle Offsets zu `kmer`, aufsteigend. Suche mit
`bisect.bisect_left(index, (kmer, -1))` die erste passende Position und sammle ab dort, solange das k-mer gleich bleibt.

*Warum `(kmer, -1)`?* Tupel werden erst nach dem ersten, dann nach dem zweiten Element verglichen. Kein Offset ist
kleiner als −1, also landet `bisect_left` vor dem ersten Eintrag mit diesem k-mer.

In [ ]:
def build_sorted_index(t: str, k: int) -> list[tuple[str, int]]:
    """All (k-mer, offset) pairs of t, sorted alphabetically by k-mer (then by offset)."""
    # Tipp: List-Comprehension mit Tupeln, danach .sort()
    raise NotImplementedError("TODO: implement build_sorted_index")


def lookup_sorted_index(index: list[tuple[str, int]], kmer: str) -> list[int]:
    """Offsets of kmer in a sorted index, found with binary search."""
    # Tipp: pos = bisect.bisect_left(index, (kmer, -1)); dann mit while weiterlaufen, solange index[pos][0] == kmer
    raise NotImplementedError("TODO: implement lookup_sorted_index")

In [ ]:
sorted_example = build_sorted_index("GTGCGTGTGGGGG", 3)
print(sorted_example)
print(lookup_sorted_index(sorted_example, "GTG"), lookup_sorted_index(sorted_example, "GGG"),
      lookup_sorted_index(sorted_example, "AAA"))

# Dictionary gegen sortierte Liste am Lambda-Genom (Werte hängen vom Rechner ab)
k = 10
rng = random.Random(1)
queries = [lambda_genome[s:s + k] for s in (rng.randint(0, len(lambda_genome) - k) for _ in range(10_000))]

start = time.perf_counter(); dict_index = build_kmer_index(lambda_genome, k); t_build_dict = time.perf_counter() - start
start = time.perf_counter(); list_index = build_sorted_index(lambda_genome, k); t_build_list = time.perf_counter() - start
start = time.perf_counter(); r1 = [dict_index.get(q, []) for q in queries]; t_query_dict = time.perf_counter() - start
start = time.perf_counter(); r2 = [lookup_sorted_index(list_index, q) for q in queries]; t_query_list = time.perf_counter() - start
assert r1 == r2
print(f"build:  dict {t_build_dict * 1000:6.1f} ms | sorted list {t_build_list * 1000:6.1f} ms")
print(f"10,000 lookups: dict {t_query_dict * 1000:6.1f} ms | sorted list {t_query_list * 1000:6.1f} ms")

In [ ]:
%%ipytest

def test_build_sorted_index_slide_example():
    assert build_sorted_index("GTGCGTGTGGGGG", 3) == [
        ("CGT", 3), ("GCG", 2), ("GGG", 8), ("GGG", 9), ("GGG", 10),
        ("GTG", 0), ("GTG", 4), ("GTG", 6), ("TGC", 1), ("TGG", 7), ("TGT", 5),
    ]


def test_lookup_sorted_index():
    index = build_sorted_index("GTGCGTGTGGGGG", 3)
    assert lookup_sorted_index(index, "GTG") == [0, 4, 6]
    assert lookup_sorted_index(index, "GGG") == [8, 9, 10]
    assert lookup_sorted_index(index, "CGT") == [3]          # erster Eintrag
    assert lookup_sorted_index(index, "TGT") == [5]          # letzter Eintrag
    assert lookup_sorted_index(index, "AAA") == []           # vor dem Anfang
    assert lookup_sorted_index(index, "TTT") == []           # hinter dem Ende
    assert lookup_sorted_index(index, "GTA") == []           # fehlt in der Mitte


def test_sorted_index_equals_dictionary_on_phix():
    g = read_genome(data_path("phix.fa"))
    dict_index = build_kmer_index(g, 6)
    list_index = build_sorted_index(g, 6)
    assert len(list_index) == len(g) - 6 + 1
    for kmer, offsets in dict_index.items():
        assert lookup_sorted_index(list_index, kmer) == offsets

## 4. Eine Hashtabelle selbst bauen (Skriptum 4.2)

Eine Hashtabelle hat eine feste Zahl von **Buckets** (Fächern). Eine **Hashfunktion** h berechnet aus jedem k-mer
die Nummer seines Buckets. Landen zwei **verschiedene** k-mers im selben Bucket, nennt man das eine **Kollision**.
Deshalb speichert man im Bucket das k-mer mit, und beim Nachschlagen werden nur die Einträge mit genau diesem k-mer
zurückgegeben.

Python-Dictionaries erledigen das intern mit einer eigenen, viel besseren Hashfunktion. Hier bauen wir zum Lernen
eine sehr einfache:

**Aufgabe 4.1:** Schreibe `kmer_to_number(kmer)`: Lies das k-mer als Zahl im Vierersystem mit A = 0, C = 1, G = 2,
T = 3. Beispiel: GTG = 2·4² + 3·4¹ + 2·4⁰ = 32 + 12 + 2 = **46**. Andere Zeichen (z. B. N) lösen `ValueError` aus.

**Aufgabe 4.2:** Schreibe `build_hash_table(t, k, n_buckets)`: eine Liste mit `n_buckets` leeren Listen. Jedes Paar
`(kmer, offset)` kommt in den Bucket `kmer_to_number(kmer) % n_buckets`. k-mers mit N werden übersprungen.

**Aufgabe 4.3:** Schreibe `lookup_hash_table(table, kmer)`: Bucket berechnen, dort nur die Offsets mit **genau
diesem** k-mer zurückgeben (aufsteigend).

*Tipp:* Horner-Schema: `value = value * 4 + digit` für jedes Zeichen von links nach rechts.

In [ ]:
BASE_TO_DIGIT = {"A": 0, "C": 1, "G": 2, "T": 3}


def kmer_to_number(kmer: str) -> int:
    """Read a DNA k-mer as a base-4 number (A=0, C=1, G=2, T=3)."""
    # Tipp: value = 0; für jede Base: value = value * 4 + BASE_TO_DIGIT[base]; unbekannte Base -> ValueError
    raise NotImplementedError("TODO: implement kmer_to_number")


def build_hash_table(t: str, k: int, n_buckets: int) -> list[list[tuple[str, int]]]:
    """Hash table with n_buckets buckets; bucket = kmer_to_number(kmer) % n_buckets."""
    # Tipp: table = [[] for _ in range(n_buckets)]  (nicht [[]] * n_buckets, warum?)
    raise NotImplementedError("TODO: implement build_hash_table")


def lookup_hash_table(table: list[list[tuple[str, int]]], kmer: str) -> list[int]:
    """Offsets of kmer: go to its bucket and keep only entries with exactly this k-mer."""
    # Tipp: Bucket-Nummer wie beim Bauen; len(table) ist die Zahl der Buckets
    raise NotImplementedError("TODO: implement lookup_hash_table")

In [ ]:
table = build_hash_table("GTGCGTGTGGGGG", 3, 7)
for number, bucket in enumerate(table):
    kmers = sorted({kmer for kmer, _ in bucket})
    note = "   <- collision" if len(kmers) > 1 else ""
    print(number, bucket, note)
print(lookup_hash_table(table, "GTG"), lookup_hash_table(table, "TGT"), lookup_hash_table(table, "AAA"))

In [ ]:
%%ipytest

def test_kmer_to_number():
    assert kmer_to_number("A") == 0
    assert kmer_to_number("T") == 3
    assert kmer_to_number("GTG") == 46
    assert kmer_to_number("TTTT") == 255
    assert kmer_to_number("ACGT") == 27
    with pytest.raises(ValueError):
        kmer_to_number("ACN")


def test_build_hash_table_small_example():
    table = build_hash_table("GTGCGTGTGGGGG", 3, 7)
    assert len(table) == 7
    assert table[3] == [("GCG", 2), ("TGT", 5)]                 # Kollision wie auf Langmeads Folie
    assert table[4] == [("GTG", 0), ("GTG", 4), ("GTG", 6)]
    assert table[5] == []
    assert sum(len(b) for b in table) == 11


def test_lookup_hash_table_handles_collisions():
    table = build_hash_table("GTGCGTGTGGGGG", 3, 7)
    assert lookup_hash_table(table, "GCG") == [2]
    assert lookup_hash_table(table, "TGT") == [5]
    assert lookup_hash_table(table, "GGG") == [8, 9, 10]
    assert lookup_hash_table(table, "AAA") == []


def test_hash_table_equals_dictionary_on_phix():
    g = read_genome(data_path("phix.fa"))
    dict_index = build_kmer_index(g, 8)
    table = build_hash_table(g, 8, 1009)
    for kmer, offsets in dict_index.items():
        assert lookup_hash_table(table, kmer) == offsets


def test_hash_table_skips_n():
    table = build_hash_table("ACNGT", 2, 5)
    assert sum(len(b) for b in table) == 2                    # nur AC und GT

## 5. Variante: nur jedes `ival`-te k-mer indexieren (Skriptum 5.1)

Langmead zeigt, dass man **nicht jedes** k-mer von T speichern muss. Speichert man nur die k-mers an geraden Offsets
(0, 2, 4, …), braucht der Index etwa halb so viel Platz. Dafür muss man beim Abfragen **zwei** k-mers von P nachschlagen:
das an Offset 0 und das an Offset 1 von P. Eines der beiden liegt immer auf einem geraden Offset von T, wenn P dort
vorkommt. Allgemein: Speichert man jedes `ival`-te k-mer, schlägt man die k-mers von P an den Offsets
0, 1, …, `ival − 1` nach.

Ein Treffer für das k-mer an Offset `j` von P beim Text-Offset `h` bedeutet: P könnte bei `h − j` beginnen. Dieser
Kandidat muss ≥ 0 sein und wird wie in Teil 2 verifiziert. Weil mehrere k-mers von P denselben Kandidaten liefern
können, sammelt man die Vorkommen in einer **Menge** (`set`).

Beispiel von Langmeads Folie (T = CGTGCGTGCTT, k = 5, nur gerade Offsets): Der Index enthält CGTGC: 0, 4; TGCGT: 2;
TGCTT: 6. Für P = GCGTGCTT steht GCGTG (Offset 0 in P) nicht im Index. CGTGC (Offset 1 in P) liefert die Treffer 0 und 4,
also die Kandidaten −1 (ungültig) und 3. Die Verifikation bestätigt Offset 3.

**Aufgabe 5.1:** Schreibe `build_sparse_index(t, k, ival)` wie `build_kmer_index`, aber nur für die Offsets
0, `ival`, 2·`ival`, …

**Aufgabe 5.2:** Schreibe `query_sparse_index(p, t, index, k, ival)`. Rückgabe `(occurrences, n_hits)` mit
aufsteigend sortierten Vorkommen und der Gesamtzahl der Index-Treffer aller nachgeschlagenen k-mers. Damit alle
`ival` k-mers von P existieren, muss `len(p) >= k + ival - 1` gelten. Sonst: `ValueError`.

*Tipp:* `range(0, len(t) - k + 1, ival)` liefert jeden `ival`-ten Offset.

In [ ]:
def build_sparse_index(t: str, k: int, ival: int) -> dict[str, list[int]]:
    """k-mer index that stores only the k-mers at offsets 0, ival, 2*ival, ..."""
    raise NotImplementedError("TODO: implement build_sparse_index")


def query_sparse_index(p: str, t: str, index: dict[str, list[int]], k: int, ival: int) -> tuple[list[int], int]:
    """Query a sparse index with the k-mers of p at offsets 0 .. ival-1. Returns (occurrences, index hits)."""
    # Tipp: for j in range(ival): Treffer h für p[j:j + k] holen; Kandidat = h - j; prüfen (>= 0, Verifikation)
    raise NotImplementedError("TODO: implement query_sparse_index")

In [ ]:
t_example = "CGTGCGTGCTT"
even_index = build_sparse_index(t_example, 5, 2)
print(even_index)
print(query_sparse_index("GCGTGCTT", t_example, even_index, 5, 2))
print(build_sparse_index(t_example, 5, 3))      # jedes dritte k-mer: Offsets 0, 3, 6

In [ ]:
%%ipytest

def test_build_sparse_index_slide_examples():
    t = "CGTGCGTGCTT"
    assert build_sparse_index(t, 5, 1) == build_kmer_index(t, 5)
    assert build_sparse_index(t, 5, 2) == {"CGTGC": [0, 4], "TGCGT": [2], "TGCTT": [6]}
    assert build_sparse_index(t, 5, 3) == {"CGTGC": [0], "GCGTG": [3], "TGCTT": [6]}


def test_query_sparse_index_slide_example():
    t = "CGTGCGTGCTT"
    assert query_sparse_index("GCGTGCTT", t, build_sparse_index(t, 5, 2), 5, 2) == ([3], 2)


def test_query_sparse_index_course_example():
    # Beispiel aus Langmeads Notebook „InvertedIndex“: jedes 4. 5-mer
    t = "CGTGCCTACTTACTTACAT"
    index = build_sparse_index(t, 5, 4)
    assert index == {"CGTGC": [0], "CCTAC": [4], "CTTAC": [8, 12]}
    # Langmeads Index liefert die Treffer 8 und 12; ab 12 ist der Text zu kurz für P, die Verifikation behält nur 8
    assert query_sparse_index("CTTACTTA", t, index, 5, 4) == ([8], 2)


def test_query_sparse_index_too_short():
    with pytest.raises(ValueError):
        query_sparse_index("ACGTAC", "ACGTACGTAC", build_sparse_index("ACGTACGTAC", 5, 3), 5, 3)


def test_query_sparse_equals_naive_on_random_dna():
    rng = random.Random(5)
    for _ in range(200):
        t = "".join(rng.choice("ACGT") for _ in range(250))
        ival = rng.randint(1, 4)
        if rng.random() < 0.7:
            start = rng.randint(0, 220)
            p = t[start:start + rng.randint(5 + ival - 1, 20)]
        else:
            p = "".join(rng.choice("AC") for _ in range(10))
        if len(p) < 5 + ival - 1:
            continue
        assert query_sparse_index(p, t, build_sparse_index(t, 5, ival), 5, ival)[0] == naive(p, t)

## 6. Messungen an phiX174 (Skriptum 6)

Die folgenden Zellen sind **fertig**. Sie verwenden deine Funktionen. Führe sie aus und beantworte danach die Fragen.

**Messung 1:** Wie hängt die Zahl der Index-Treffer von k ab? Gesucht werden die 1000 Reads (je 100 nt) aus Woche 6,
nur der gespeicherte Strang. Echte Vorkommen gibt es 7 (wie in Woche 7).

In [ ]:
print(f"{'k':>3} {'distinct':>9} {'4^k':>12} {'index hits':>11} {'occurrences':>12}")
for k in [4, 6, 8, 10, 12, 16]:
    index = build_kmer_index(genome, k)
    hits, occurrences = 0, 0
    for read in reads:
        occ, n_hits = query_kmer_index(read, genome, index, k)
        hits += n_hits
        occurrences += len(occ)
    print(f"{k:>3} {len(index):>9} {4 ** k:>12} {hits:>11} {occurrences:>12}")

**Messung 2:** Naiver Algorithmus gegen Index (k = 8): Laufzeit für die 1000 Reads. Die Werte hängen vom Rechner ab.

In [ ]:
start = time.perf_counter()
naive_result = [naive(read, genome) for read in reads]
t_naive = time.perf_counter() - start

start = time.perf_counter()
index8 = build_kmer_index(genome, 8)
t_build = time.perf_counter() - start

start = time.perf_counter()
index_result = [query_kmer_index(read, genome, index8, 8)[0] for read in reads]
t_query = time.perf_counter() - start

assert naive_result == index_result
print(f"naive: {t_naive * 1000:.0f} ms | build index: {t_build * 1000:.1f} ms | 1000 queries: {t_query * 1000:.1f} ms")

**Messung 3:** Weniger Einträge, mehr Abfragen: der Index mit jedem `ival`-ten 8-mer.

In [ ]:
print(f"{'ival':>4} {'entries':>8} {'lookups/read':>13} {'index hits':>11} {'occurrences':>12}")
for ival in [1, 2, 3, 4]:
    index = build_sparse_index(genome, 8, ival)
    entries = sum(len(v) for v in index.values())
    hits, occurrences = 0, 0
    for read in reads:
        occ, n_hits = query_sparse_index(read, genome, index, 8, ival)
        hits += n_hits
        occurrences += len(occ)
    print(f"{ival:>4} {entries:>8} {ival:>13} {hits:>11} {occurrences:>12}")

**Messung 4:** Warum scheitern so viele Index-Treffer an der Verifikation? Laut NCBI ist das phiX174-Genom **ringförmig**
(circular). In der FASTA-Datei ist der Ring an einer Stelle aufgeschnitten. Ein Read, der über diese Stelle läuft,
beginnt am Ende der Datei und setzt sich am Anfang fort. Hängt man die ersten 99 Basen hinten an, kann die
Verifikation solche Reads bestätigen.

In [ ]:
index16 = build_kmer_index(genome, 16)
near_end = sum(1 for read in reads for h in index16.get(read[:16], []) if h + len(read) > len(genome))
print(f"index hits (k=16) where the read would run past the end of the FASTA sequence: {near_end}")

circular_genome = genome + genome[:99]          # Ringschluss für Reads mit 100 nt
index_circular = build_kmer_index(circular_genome, 16)
found_linear = sum(1 for read in reads if query_kmer_index(read, genome, index16, 16)[0])
found_circular = sum(1 for read in reads if query_kmer_index(read, circular_genome, index_circular, 16)[0])
print(f"reads found: linear genome {found_linear} | circular genome {found_circular}")

In [ ]:
%%ipytest

def test_measurement_4_circular_genome():
    g = read_genome(data_path("phix.fa"))
    seqs, _ = read_fastq(data_path("ERR266411_1.first1000.fastq"))
    circular = g + g[:99]
    index = build_kmer_index(circular, 16)
    assert sum(1 for s in seqs if query_kmer_index(s, circular, index, 16)[0]) == 408


def test_measurement_1_hits_k8_and_k16():
    g = read_genome(data_path("phix.fa"))
    seqs, _ = read_fastq(data_path("ERR266411_1.first1000.fastq"))
    for k, expected_hits in [(8, 600), (16, 487)]:
        index = build_kmer_index(g, k)
        results = [query_kmer_index(s, g, index, k) for s in seqs]
        assert sum(n for _, n in results) == expected_hits
        assert sum(len(occ) for occ, _ in results) == 7


def test_measurement_3_sparse_entries():
    g = read_genome(data_path("phix.fa"))
    assert [sum(len(v) for v in build_sparse_index(g, 8, ival).values()) for ival in (1, 2, 3, 4)] == \
        [5379, 2690, 1793, 1345]

## 7. Nachdenken und notieren

Beantworte die Fragen mit deinen Messergebnissen und dem Skriptum. Schreibe die Antworten in die Zelle darunter.

1. Warum gibt es bei k = 4 viel mehr Index-Treffer als bei k = 12, obwohl beide Male nur 7 Vorkommen herauskommen?
2. Ab k = 10 sinkt die Zahl der Index-Treffer kaum noch, bleibt aber weit über 7. Woran liegt das? Nutze Messung 4.
3. Dictionary oder sortierte Liste: Welche Variante war beim Nachschlagen schneller? Nenne einen Vorteil der
   sortierten Liste.
4. Der Index mit `ival = 4` hat nur ein Viertel der Einträge. Was kostet diese Ersparnis beim Abfragen?
5. Ist die Suche mit einem k-mer-Index ein online- oder ein offline-Verfahren? Begründe mit Woche 7.

*Deine Antworten:*

1.
2.
3.
4.
5.

## Bonus: Subsequenz-Index (Skriptum 5.2, optional)

Eine **Subsequenz** besteht aus Zeichen von S in derselben Reihenfolge, die aber nicht nebeneinander stehen müssen.
Langmeads Beispiel: `seq = "AACCGGTT"`, dann ist `seq[0] + seq[1] + seq[5] + seq[7]` = AAGT eine Subsequenz, aber kein
Teilstring. Ein **Subsequenz-Index** speichert statt k-mers solche Subsequenzen. Welche Positionen genommen werden,
legt eine Maske fest, z. B. `mask = [0, 1, 3, 5, 6]`: ab Offset i die Zeichen i, i+1, i+3, i+5, i+6.

Langmeads Beispiel mit T = CGTGCGTGCTT und dieser Maske: CGGGT: 0, GTCTG: 1, TGGGC: 2, GCTCT: 3, CGGTT: 4.
Für P = GCGTACT ergibt die Maske GCTCT; der Index liefert Offset 3. In T steht dort GCGTGCT: P unterscheidet sich nur an
Position 4, die die Maske nicht ansieht. Ein Subsequenz-Index findet so Kandidaten **trotz eines Mismatches**. Das ist
die Idee hinter den „spaced seeds“ von PatternHunter (Skriptum 7).

**Bonus 1:** `subsequence(s, offset, mask)` liefert den String aus `s[offset + m]` für jedes `m` in `mask`.
**Bonus 2:** `build_subseq_index(t, mask)`: Dictionary Subsequenz → Offsets für alle Offsets
`0 .. len(t) - (max(mask) + 1)`.
**Bonus 3:** `query_subseq_index(p, index, mask)`: die Index-Treffer für die Subsequenz von P ab Offset 0
(ohne Verifikation, P muss mindestens `max(mask) + 1` Zeichen haben).

In [ ]:
def subsequence(s: str, offset: int, mask: list[int]) -> str:
    """Characters s[offset + m] for every m in mask, joined to a string."""
    raise NotImplementedError("Bonus: implement subsequence")


def build_subseq_index(t: str, mask: list[int]) -> dict[str, list[int]]:
    """Map every masked subsequence of t to the offsets where it starts."""
    raise NotImplementedError("Bonus: implement build_subseq_index")


def query_subseq_index(p: str, index: dict[str, list[int]], mask: list[int]) -> list[int]:
    """Index hits (candidate offsets) for the masked subsequence of p at offset 0."""
    raise NotImplementedError("Bonus: implement query_subseq_index")

In [ ]:
%%ipytest

MASK = [0, 1, 3, 5, 6]


def test_subsequence_langmead_example():
    assert subsequence("AACCGGTT", 0, [0, 1, 5, 7]) == "AAGT"
    assert "AACCGGTT".find("AAGT") == -1                  # Subsequenz, aber kein Teilstring


def test_build_subseq_index_slide_example():
    assert build_subseq_index("CGTGCGTGCTT", MASK) == {
        "CGGGT": [0], "GTCTG": [1], "TGGGC": [2], "GCTCT": [3], "CGGTT": [4],
    }


def test_query_subseq_index_finds_candidate_despite_mismatch():
    index = build_subseq_index("CGTGCGTGCTT", MASK)
    assert query_subseq_index("GCGTACT", index, MASK) == [3]
    assert query_subseq_index("AAAAAAA", index, MASK) == []